# Figure 5 — Robustness of the primary weather–crash associations across specifications

- **FIG_ID**: `Figure 5`（编号真源：`data/figure_numbering.csv`，§4.0；output stem `Figure_5`）
- **图位**: v6 正文行 244（§4.4）；对应 `[FIGURE 5 PLACEHOLDER]`
- **数据依赖**: `data/fig05_robustness_rows.csv`（A 层冻结，26 行；SHA-256 见 `data/figure_data_manifest.json`）
  —— 长表：每行一个 panel × 条目，字段含 estimate / ci_low / ci_high / ci_method / provenance
- **上游变更提示**: 若 Table 3 系列、阈值导向管线表或 G1 bootstrap 复制文件更新，先运行
  `python scripts/extract_fig05_data.py` 重新冻结（Cell 3 会拦截哈希不匹配）
- **输出**: `out/draft/Figure_5.{png,pdf,eps}`（`FIG_DRAFT=1`，默认）/ `out/submission/Figure_5.*`
  （`FIG_DRAFT=0`）+ `out/captions/Figure_5_caption.txt` + `qc/figure_5_qc.json`
- **图面约束**（v6 行 244 + 计划书 §2-Fig5）: 每个暴露一个 panel（(a) 低能见度 / (b) 连续降水 /
  (c) 强风与冰冻）；**panel (c) 内再分「强风 / 冰冻」两个子列**，两类暴露不叠加；全图统一行序，
  每行一种规格，点估计 + 95% 区间，参考线 IRR = 1.0
  - 行 1–2 自匹配设计的两个拟合版本（仅降水）；行 3 主规格；行 4 替代协变量规格（仅低能见度）；
    **行 5「替代时间控制」已按 G2 判定删除**（`T103_period_stability.csv` 改属 Figure S8）；
    行 6 阈值导向管线（每个阈值点带短标签；低能见度无可比估计 → 图内写 `No comparable estimate`）；
    行 7「Inference-method check」（G1 已关闭，五符号并列；标签由图例承担）
- **设计系统（§1.6，v2.3.1）**: 主色 NJU purple `#6E0F6D`（主要规格 + 预先指定的次级）；
  预注册阈值管线用辅助色 `#D89000`；推断方法核验用灰系 `#7A7A7A` / `#4A4A4A`；
  参考线深灰虚线；行 3 用**极浅**底带 `#F1EAF1` 强调（不代表置信区间）；**必须带图例说明各层含义**；
  图内禁用内部代号
- **版面纪律**: 行标签写在坐标区内部左缘（不用 y 轴刻度标签），成图宽度因此可稳定压在 180 mm；
  改动字号/标签/图例后必须用 `python scripts/fig05_size_calibration.py` 重新标定 FIG_W 与 MARGINS
- **门槛纪律**: 第 7 组的 bootstrap 区间**必须**由 `g1_upstream_exact_B999.csv` 的 926 条有效复制
  现场重算（百分位法）；数据可得性声明写 926，**不得写 934**（上游日志口径）；
  任何中断或未验证的结果不得出现在本图（§4.4、§7-J）


In [ ]:
%load_ext autoreload
%autoreload 2

import os
import sys
import time
from pathlib import Path


def _resolve_scripts_dir() -> Path:
    """定位 figure/scripts，使本册在**任何内核 cwd** 下都可运行（§1.5.3）。"""
    here = Path.cwd()
    cands = [here / "scripts", here.parent / "scripts"]
    cands += [p / "figure" / "scripts" for p in (here, *here.parents)]
    cands.append(Path(r"D:\NJU_folder\final_update\15_paper\figure\scripts"))
    for c in cands:
        if (c / "fig_common.py").exists():
            return c.resolve()
    raise RuntimeError("未找到 figure/scripts/fig_common.py：请从本仓库、figure/ 或 "
                       "figure/notebooks/ 任一目录启动内核，或检查仓库路径")


SCRIPTS_DIR = _resolve_scripts_dir()
if str(SCRIPTS_DIR) not in sys.path:
    sys.path.append(str(SCRIPTS_DIR))
print("scripts dir:", SCRIPTS_DIR)

from fig_common import *          # 样式、色板、导出、核验（唯一来源，勿在本册硬编码样式）
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

print(f">>> Figure 5 notebook | design system {DESIGN_VERSION} | notebook API v{NOTEBOOK_API} "
      f"| 2x2 四 panel 规格曲线（行标签仅左列）+ 行 7 五符号核验")
print(">>> 若本行未出现、或版本号低于上面的值，说明内核里跑的是**旧版 notebook**："
      "请在编辑器执行 Revert File（VS Code）/ 关闭标签后重新打开本文件")

# ---- 本图配置 -------------------------------------------------------------
FIG_ID = "Figure 5"
IS_DRAFT = os.environ.get("FIG_DRAFT", "1") == "1"   # 1 -> out/draft；0 -> out/submission
DATA_FILE = "fig05_robustness_rows.csv"
t0 = time.time()
meta = figure_meta(FIG_ID)
print(f"FIG_ID={FIG_ID}  stem={meta['filename_stem']}  IS_DRAFT={IS_DRAFT}")
print("data dir   :", FIG_DATA, "|", DATA_FILE, "存在:", (FIG_DATA / DATA_FILE).exists())
print("输出目录:", FIG_OUT_DRAFT if IS_DRAFT else FIG_OUT_SUB)


In [ ]:
# ---- 数据加载与核验（不匹配即中断）---------------------------------------
report = verify_figure_data(FIG_ID, data_files=[DATA_FILE])
df = pd.read_csv(FIG_DATA / DATA_FILE, encoding="utf-8-sig")

# 结构与门槛自检（与计划书 §2-Fig5 的最终行集逐项对齐）
# 注意：CSV 中的空 subpanel 读入后是 NaN，故先 fillna("")
df["subpanel"] = df.subpanel.fillna("")
assert set(df.panel) == {"a", "b", "c"}, f"panel 应只有 a/b/c，实际 {sorted(set(df.panel))}"
assert set(df[df.panel == "c"].subpanel) == {"gust", "freezing"}, (
    "panel (c) 必须分成 gust / freezing 两个子列（两类暴露分列显示，不叠加）")
assert set(df[df.panel != "c"].subpanel) == {""}, "只有 panel (c) 有子列"
assert 5 not in set(df.row_order), (
    "第 5 组（替代时间控制）已按 G2 判定删除；若数据里又出现行 5，说明取数脚本被回退——"
    "请重跑 python figure\\scripts\\extract_fig05_data.py")
assert set(df.row_order) == {1, 2, 3, 4, 6, 7}, f"行序应为 1/2/3/4/6/7，实际 {sorted(set(df.row_order))}"

# G1 门槛：行 7 的 bootstrap 必须是 926 条留存复制的百分位区间
BOOT_REG = {"a": (1.0971, 1.3429), "c": (0.9827, 1.2190)}
BOOT_FRZ = (0.8316, 1.4655)
boot = df[df.ci_method.astype(str).str.startswith("Section-cluster (pairs) bootstrap")]
assert len(boot) == 3, f"bootstrap 应覆盖低能见度/强风/冰冻三行，实际 {len(boot)}"
for _, r in boot.iterrows():
    reg = BOOT_FRZ if "freezing" in str(r.exposure) else BOOT_REG[r.panel]
    assert abs(r.ci_low - reg[0]) < 5e-4 and abs(r.ci_high - reg[1]) < 5e-4, (
        f"{r.exposure} 的 bootstrap 区间 {r.ci_low, r.ci_high} 与登记值 {reg} 不符")

# 登记值自检（计划书 §4.3 关联部分）
REG = {("b", 1): (1.088, 1.038, 1.140), ("b", 2): (1.121, 1.069, 1.176),
       ("a", 3): (1.200, 1.086, 1.326), ("c", 3): (1.119, 1.001, 1.252),
       ("a", 4): (1.107, 1.007, 1.216), ("b", 6): (1.475, 1.254, 1.735)}
for (p, o), (e, lo, hi) in REG.items():
    rows = df[(df.panel == p) & (df.row_order == o)]
    if o == 6 and p == "b":
        rows = rows[rows.estimate > 1.4]
    r = rows.iloc[0]
    assert max(abs(r.estimate - e), abs(r.ci_low - lo), abs(r.ci_high - hi)) < 5e-3,         f"({p}) 行 {o} 与登记值不符：{r.estimate, r.ci_low, r.ci_high} vs {e, lo, hi}"

# 纪律：每个绘制条目必须登记区间来源；无可比估计的条目不落点
assert df[df.plotted].provenance.astype(str).str.strip().ne("").all(), "存在未登记区间来源的绘制条目"
assert not df[df.status == "not_robust"].plotted.any(), "无可比估计的条目不得绘制标记"
# 图内禁用内部代号
for bad in ("NB2", "SPEC_P", "SPEC_F", "M3", "M4"):
    hit = df[["exposure", "row_label", "panel_exposure"]].astype(str).apply(
        lambda c: c.str.contains(bad, regex=False)).any().any()
    assert not hit, f"图内字段出现内部代号 {bad}"
print("核验通过：", report["checked"])
df[["panel", "subpanel", "row_order", "exposure", "estimate", "ci_low", "ci_high",
    "ci_method", "status"]]


In [ ]:
# ---- 绘图：2x2 四 panel 规格曲线（2026-10-02 改版：行标签仅左列，图顶一行估计器图例）----------
# 面板装配放在同图模块 fig05_robustness_core.build_figure() 内，
# 使命令行试绘（--draft/--submission）与本册出图共用唯一实现，避免两处拷贝漂移。
from fig05_robustness_core import (build_figure, PANELS, ROW_LABEL_SHORT,
                                   _estimator_key, ESTIMATOR_LABEL)

fig = build_figure(df)
for i, ax in enumerate(fig.get_axes()):
    bb = ax.get_position()
    print(f"panel {i} x0={bb.x0:.3f} y0={bb.y0:.3f} w={bb.width:.3f} h={bb.height:.3f} (figure fraction)")
for lg in fig.legends:
    ttl = lg.get_title()
    print("legend:", (str(ttl) if ttl else "") or "(no title)", "|", [t.get_text() for t in lg.get_texts()])
print("panels:", [(k, s or None) for (k, _, _, s) in PANELS])
print("row labels (shared left column):", [ROW_LABEL_SHORT[o] for o in (1, 2, 3, 4, 6)],
      "+ Inference-method check (inference block, five fixed y positions)")
print("row 7 estimator symbols:", [ESTIMATOR_LABEL[k] for k in
                                  [_estimator_key(e) for e in df[df.row_order == 7].exposure.unique()]])
fig


In [ ]:
# ---- 导出双版本 + caption + qc -------------------------------------------
written = export_figure(fig, FIG_ID, is_draft=IS_DRAFT)
print("\n".join(str(p) for p in written))

from fig05_robustness_core import CAPTION

cap = write_caption(
    FIG_ID, CAPTION,
    title="Figure 5. Robustness of the primary weather-crash associations across specifications.")
print(cap)

qc = fig_qc_dump(FIG_ID, {
    "status": "ok", "is_draft": IS_DRAFT, "seconds": round(time.time() - t0, 1),
    "data_checked": report["checked"], "data_file": DATA_FILE,
    "rows_shown": sorted(set(int(o) for o in df.row_order)),
    "row_5_status": "deleted by G2 decision (alternative temporal control not established)",
    "palette": ("NJU purple #6E0F6D (primary + prespecified secondary); accent #D89000 "
                "(threshold-oriented pipeline); grey #7A7A7A (interval-estimator verification)"),
    "layout": "four plotting columns (a low visibility / b precipitation / c strong wind / c freezing), "
              "unified row order, row labels at the axes left edge, row 6 points carry threshold labels, "
              "row 7 stacks five estimator symbols within the row",
    "outputs": [{"path": str(p), "bytes": p.stat().st_size, "sha256": sha256_file(p)} for p in written],
    "caption": str(cap),
})
print(qc)

# 投稿版凭据：只在非草稿运行时写入，避免被后续草稿重跑覆盖（发布门禁以它为准）
if not IS_DRAFT:
    rec = fig_submission_record(FIG_ID, written, cap,
                                {"data_checked": report["checked"], "data_file": DATA_FILE})
    print("submission record:", rec)
